# Optimal Executable Process Plan Generation for Capability-Constrained Modular Plants

## Overview

This notebook is the reproducibility-oriented companion to the paper *Optimal Executable Process Plan Generation for Capability-Constrained Modular Plants*. It instantiates the HC10-HC40 reference plant discussed in the manuscript and follows the complete workflow from plant description and ISA-88 recipe generation to reachable-state construction and optimized execution-path selection.

The notebook operationalizes the three research questions of the paper:

- **RQ1**: decide whether a candidate operation is legal under the current plant state, topology, capability set, and recipe progress;
- **RQ2**: control the branching factor induced by partial transfers through symbolic transfer states, lazy instantiation, and optional concrete transfer-volume enumeration; and
- **RQ3**: select one globally preferred executable sequence from the reachable-state graph.

The implementation is organized around the same four ingredients emphasized in the paper:

1. **FSA-style legality filtering** over plant capabilities, interfaces, capacities, and recipe progress;
2. **partial-transfer state generation** with either deferred symbolic resolution or concrete transfer-volume enumeration;
3. **parallel BFS reachable-state graph construction** over immutable state representations; and
4. **OPT / Pyomo-based path optimization**, where **OPT** denotes the **optimizer / optimization-solving stage** that selects one preferred executable path on the reachable-state graph.

The default execution path reproduces the fixed four-module reference case. Seeded randomized generation is retained as a supplementary branch for batch experiments and infeasible-case studies, but it is not the primary narrative of this notebook.

By default, `use_lazy_transfer_states = True`, so partial transfers stay symbolic until a downstream rule requires a concrete amount. When `use_lazy_transfer_states = False`, the planner instead enumerates concrete transfer quantities at multiples of `min_transfer_volume` and additionally includes the full admissible transfer amount, which can substantially increase the reachable-state graph.


## Notebook Roadmap

The notebook follows the same high-level progression as the paper, from reference-case specification to executable-plan extraction. The links below provide a compact table of contents for the main implementation layers and outputs.

1. [Reference Plant Configuration](#Reference-Plant-Configuration)
2. [General Recipe Construction](#General-Recipe-Construction)
3. [FSA-Style Legality Filtering](#FSA-Style-Legality-Filtering)
4. [Lazy Instantiation of Symbolic Transfer States](#Lazy-Instantiation-of-Symbolic-Transfer-States)
5. [Parallel BFS Reachable-State Graph Construction](#Parallel-BFS-Reachable-State-Graph-Construction)
6. [OPT over the Reachable-State Graph](#OPT-over-the-Reachable-State-Graph)
7. [Operation Sequence Extraction](#Operation-Sequence-Extraction)
8. [Structured Artifact Export](#Structured-Artifact-Export)

Within this roadmap, the legality layer corresponds to the paper's FSA-style admissibility filtering, the BFS section constructs the reachable-state graph, and the Pyomo model realizes the **OPT stage**, meaning the **optimizer / optimization-solving layer** used to select one preferred executable sequence.


## Runtime Environment and Reproducibility Notes

This notebook is intended to be executed from top to bottom.

- The default configuration uses the paper-aligned HC10-HC40 reference plant and reference order.
- The literal setting `use_original_modplants = True` keeps the fixed reference case active.
- When the flag is switched to `False`, the supplementary generator uses `MODPLANT_SEED` and `random_seed` to produce deterministic randomized instances.
- `use_lazy_transfer_states = True` keeps lazy symbolic transfer-state instantiation enabled by default.
- `min_transfer_volume = 1.0` controls concrete transfer enumeration when lazy transfer states are disabled.
- `MODPLANT_USE_LAZY_TRANSFER_STATES` and `MODPLANT_MIN_TRANSFER_VOLUME` can override these transfer settings during non-interactive runs.
- `batch_runner.ipynb` extracts the code cells of this notebook, strips notebook magics, and reuses the same execution path for seeded runs.
- Generated XML recipes, parsed JSON recipes, result corpora, and logs are written into the repository runtime directories.


In [69]:
import os
import sys

repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.append(repo_root)


## Imports and Notebook Utilities

The next cell resets the interactive namespace, enables autoreload for local helper modules, imports the shared scientific and optimization dependencies, and defines a lightweight DataFrame display helper used throughout the notebook.


In [70]:
%reset -f
%load_ext autoreload
%autoreload 2

from pyomo.environ import *
import pandas as pd
from collections import deque
from types import SimpleNamespace

try:
    from IPython.display import Markdown, display
except Exception:
    display = None
    Markdown = None

def display_dataframe_to_user(name=None, dataframe=None):
    if dataframe is None:
        return None
    if display is not None:
        if name and Markdown is not None:
            display(Markdown(f"**{name}**"))
        display(dataframe)
    else:
        if name:
            print(name)
        try:
            print(dataframe)
        except Exception:
            pass
    return dataframe

tools = SimpleNamespace(display_dataframe_to_user=display_dataframe_to_user)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Reference Plant Configuration

The next cell defines the fixed HC10-HC40 configuration used in the paper-facing reference case. It materializes four dictionaries that together form the internal plant model:

- `modplant_ops` for supported operations, operation parameters, and cost weights;
- `modplant_interfaces` for directed input and output ports;
- `modplant_maximum_volume` for vessel capacities; and
- `modplant_resources` for the initial material allocation.

These structures provide the capability, topology, and inventory information consumed later by legality filtering, state expansion, and path optimization.


In [71]:
modplant_ops = {
    'HC10': [('Draining', 0.1, 3), ('Filling', 0.1, 0), ('Settling', '', 1), ('Stirring', '100', 3), ('Stirring', '200', 3),('Connect', '', 1), ('Disconnect', '', 0),  ('None', '', 0)],
    'HC20': [('Draining', 0.1, 3), ('Filling', 0.1, 0), ('Settling', '', 1), ('Stirring', '150', 3), ('Stirring', '300', 3), ('Connect', '', 1), ('Disconnect', '', 0), ('None', '', 0)],
    'HC30': [('Draining', 0.1, 3), ('Filling', 0.1, 0), ('Settling', '', 1), ('Stirring', '100', 3), ('Stirring', '150', 3),('Connect', '', 1), ('Disconnect', '', 0),  ('None', '', 0)],
    'HC40': [('Draining', 0.1, 3), ('Filling', 0.1, 0), ('Settling', '', 1), ('Connect', '', 1), ('Disconnect', '', 0), ('None', '', 0)]
}

modplant_interfaces = {
    'HC10': [('Input', 'HC10_In1'), ('Input', 'HC10_In2'), ('Input', 'HC10_In3'), ('Output', 'HC10_Out1'), ('Output', 'HC10_Out2'), ('Output', 'HC10_Out3')],
    'HC20': [('Input', 'HC20_In1'), ('Input', 'HC20_In2'), ('Input', 'HC20_In3'), ('Output', 'HC20_Out1'), ('Output', 'HC20_Out2'), ('Output', 'HC20_Out3')],
    'HC30': [('Input', 'HC30_In1'), ('Input', 'HC30_In2'), ('Input', 'HC30_In3'), ('Input', 'HC30_In4'), ('Output', 'HC30_Out1')],
    'HC40': [('Input', 'HC40_In1'), ('Output', 'HC40_Out1'), ('Output', 'HC40_Out2')],
}

modplant_maximum_volume = {
    'HC10': [10],
    'HC20': [15],
    'HC30': [10],
    'HC40': [30]
}

modplant_resources = {
    'HC10': ['A', 10],
    'HC20': ['B', 10],
    'HC30': ['C', 10],
}


## Optional Seeded Randomized Configuration (Supplementary)

The paper narrative uses the fixed reference plant. The following cell preserves the seeded randomized generator used for supplementary sweeps and capability-mismatch studies. The default remains `use_original_modplants = True`; when the flag is flipped to `False`, `MODPLANT_SEED` and `random_seed` make the generated plant description deterministic for batch execution.

The same configuration cell also exposes `use_lazy_transfer_states` and `min_transfer_volume`. Lazy symbolic transfer states remain enabled by default. When lazy mode is disabled, the planner enumerates concrete transfer quantities at multiples of `min_transfer_volume` and appends the full admissible transfer amount, which can increase the number of generated states substantially. `MODPLANT_USE_LAZY_TRANSFER_STATES` and `MODPLANT_MIN_TRANSFER_VOLUME` provide matching environment-variable overrides for batch execution.


In [72]:
import os
import random

def _parse_env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    normalized = raw.strip().lower()
    if normalized in {"1", "true", "yes", "y", "on"}:
        return True
    if normalized in {"0", "false", "no", "n", "off"}:
        return False
    raise ValueError(
        f"Environment variable {name} must be one of 1/0, true/false, yes/no, on/off; got {raw!r}"
    )

def _parse_env_float(name: str, default: float) -> float:
    raw = os.getenv(name)
    if raw is None:
        return default
    try:
        return float(raw)
    except ValueError as exc:
        raise ValueError(f"Environment variable {name} must be a float; got {raw!r}") from exc

use_original_modplants = True
use_lazy_transfer_states = _parse_env_bool('MODPLANT_USE_LAZY_TRANSFER_STATES', True)
min_transfer_volume = _parse_env_float('MODPLANT_MIN_TRANSFER_VOLUME', 1.0)
if min_transfer_volume <= 0:
    raise ValueError(f"min_transfer_volume must be > 0; got {min_transfer_volume}")

env_seed = os.getenv('MODPLANT_SEED')
random_seed = int(env_seed) if env_seed is not None else random.randint(1, 10000)
min_modplant_count = 4
max_modplant_count = 4

rate_choices = [0.1, 0.2, 0.3]
rpm_choices = list(range(50, 301, 50))

def generate_random_modplants(seed: int, min_n: int, max_n: int):
    random.seed(seed)
    n = random.randint(max(4, min_n), min(6, max_n))
    names = [f"HC{(i + 1) * 10}" for i in range(n)]

    w_ops = {}
    w_ifaces = {}
    w_cap = {}

    settling_assigned = False
    stirring_assigned = False

    for name in names:
        num_inputs = random.randint(1, 4)
        num_outputs = random.randint(1, 4)
        w_ifaces[name] = [
            ("Input", f"{name}_In{i + 1}") for i in range(num_inputs)
        ] + [
            ("Output", f"{name}_Out{i + 1}") for i in range(num_outputs)
        ]

        w_cap[name] = [random.choice(range(10, 31, 5))]

        drain_rate = random.choice(rate_choices)
        fill_rate = random.choice(rate_choices)
        ops = [
            ("Draining", drain_rate, 3),
            ("Filling", fill_rate, 0),
            ("Connect", "", 1),
            ("Disconnect", "", 0),
            ("None", "", 0),
        ]

        if not settling_assigned or random.random() < 0.5:
            ops.append(("Settling", "", random.randint(1, 3)))
            settling_assigned = True

        for rpm in random.sample(rpm_choices, random.randint(1, 2)):
            ops.append(("Stirring", str(rpm), random.randint(1, 4)))
        stirring_assigned = stirring_assigned or any(op[0] == "Stirring" for op in ops)

        w_ops[name] = ops

    if not settling_assigned:
        w_ops[names[0]].append(("Settling", "", random.randint(1, 3)))
    if not stirring_assigned:
        w_ops[names[0]].append(("Stirring", str(random.choice(rpm_choices)), random.randint(1, 4)))

    resources = {}
    for material, wb in zip(["A", "B", "C"], random.sample(names, k=min(3, len(names)))):
        cap = w_cap[wb][0]
        resources[wb] = [material, min(random.randint(5, 15), cap)]

    return w_ops, w_ifaces, w_cap, resources

print(f"use_lazy_transfer_states = {use_lazy_transfer_states}")
print(f"min_transfer_volume = {min_transfer_volume}")

if not use_original_modplants:
    modplant_ops, modplant_interfaces, modplant_maximum_volume, modplant_resources = generate_random_modplants(
        seed=random_seed,
        min_n=min_modplant_count,
        max_n=max_modplant_count,
    )


## Tabular Plant Summary

Before recipe generation starts, the notebook flattens the active plant description into a summary table. This inspection view is useful both for interactive verification in the notebook and for ASCII logging in batch runs.


In [73]:
summary_rows = []
for wb in sorted(modplant_ops):
    ifaces = modplant_interfaces.get(wb, [])
    num_inputs = sum(1 for t, _ in ifaces if t == 'Input')
    num_outputs = sum(1 for t, _ in ifaces if t == 'Output')
    max_vols = modplant_maximum_volume.get(wb, [])
    max_vol = max_vols[0] if max_vols else None
    res = modplant_resources.get(wb)
    res_str = f"{res[0]}:{res[1]}" if res else ''

    ops = modplant_ops.get(wb, [])
    if not ops:
        summary_rows.append({
            'ModPlant': wb,
            'Inputs': num_inputs,
            'Outputs': num_outputs,
            'MaxVolume': max_vol,
            'Resources': res_str,
            'Operation Type': '',
            'Operation Param': '',
            'Operation Cost': '',
        })
    else:
        for op_type, op_param, op_cost in ops:
            summary_rows.append({
                'ModPlant': wb,
                'Inputs': num_inputs,
                'Outputs': num_outputs,
                'MaxVolume': max_vol,
                'Resources': res_str,
                'Operation Type': op_type,
                'Operation Param': op_param,
                'Operation Cost': op_cost,
            })

try:
    df_modplant_summary = pd.DataFrame(summary_rows)
    _ = tools.display_dataframe_to_user(name='ModPlant Summary (Expanded)', dataframe=df_modplant_summary)
    print('ModPlant Summary (Expanded):')
    try:
        print(df_modplant_summary.to_string(index=False))
    except Exception:
        for row in summary_rows:
            print(row)
    try:
        import gc
        del df_modplant_summary
        gc.collect()
    except Exception:
        pass
except Exception as exc:
    print(f'Unable to render ModPlant summary: {exc}')


**ModPlant Summary (Expanded)**

,ModPlant,Inputs,Outputs,MaxVolume,Resources,Operation Type,Operation Param,Operation Cost
0,HC10,3,3,10,A:10,Draining,0.1,3
1,HC10,3,3,10,A:10,Filling,0.1,0
2,HC10,3,3,10,A:10,Settling,,1
3,HC10,3,3,10,A:10,Stirring,100,3
4,HC10,3,3,10,A:10,Stirring,200,3
5,HC10,3,3,10,A:10,Connect,,1
6,HC10,3,3,10,A:10,Disconnect,,0
7,HC10,3,3,10,A:10,None,,0
8,HC20,3,3,15,B:10,Draining,0.1,3
9,HC20,3,3,15,B:10,Filling,0.1,0


ModPlant Summary (Expanded):
ModPlant  Inputs  Outputs  MaxVolume Resources Operation Type Operation Param  Operation Cost
    HC10       3        3         10      A:10       Draining             0.1               3
    HC10       3        3         10      A:10        Filling             0.1               0
    HC10       3        3         10      A:10       Settling                               1
    HC10       3        3         10      A:10       Stirring             100               3
    HC10       3        3         10      A:10       Stirring             200               3
    HC10       3        3         10      A:10        Connect                               1
    HC10       3        3         10      A:10     Disconnect                               0
    HC10       3        3         10      A:10           None                               0
    HC20       3        3         15      B:10       Draining             0.1               3
    HC20       3        3      

## ModPlant Operations and Costs

The following table expands `modplant_ops` into one row per supported operation. It is the compact inspection view of the capability and cost layer that is later queried during candidate generation and optimization.


In [74]:
df_ops = pd.DataFrame([
    {"ModPlant": wb, "Operation Type": op[0], "Operation Param": op[1], "Cost": op[2]}
    for wb, op_list in modplant_ops.items()
    for op in op_list
])

_ = tools.display_dataframe_to_user(name="ModPlant Operations", dataframe=df_ops)

try:
    import gc
    del df_ops
    gc.collect()
except Exception:
    pass


**ModPlant Operations**

,ModPlant,Operation Type,Operation Param,Cost
0,HC10,Draining,0.1,3
1,HC10,Filling,0.1,0
2,HC10,Settling,,1
3,HC10,Stirring,100,3
4,HC10,Stirring,200,3
5,HC10,Connect,,1
6,HC10,Disconnect,,0
7,HC10,None,,0
8,HC20,Draining,0.1,3
9,HC20,Filling,0.1,0


## ModPlant I/O Interfaces

The following cell flattens the active `modplant_interfaces` mapping into a tabular view. It does not synthesize interfaces from an auxiliary source; it simply exposes the directed port set that the connection logic uses during search.


In [75]:
df_interfaces = pd.DataFrame([
    {"ModPlant": wb, "Interface Type": iface[0], "Interface Name": iface[1]}
    for wb, iface_list in modplant_interfaces.items()
    for iface in iface_list
])

_ = tools.display_dataframe_to_user(name="ModPlant Interfaces", dataframe=df_interfaces)

try:
    import gc
    del df_interfaces
    gc.collect()
except Exception:
    pass


**ModPlant Interfaces**

,ModPlant,Interface Type,Interface Name
0,HC10,Input,HC10_In1
1,HC10,Input,HC10_In2
2,HC10,Input,HC10_In3
3,HC10,Output,HC10_Out1
4,HC10,Output,HC10_Out2
5,HC10,Output,HC10_Out3
6,HC20,Input,HC20_In1
7,HC20,Input,HC20_In2
8,HC20,Input,HC20_In3
9,HC20,Output,HC20_Out1


## ModPlant Maximum Volumes

This table exposes the vessel-capacity model stored in `modplant_maximum_volume`. In the current artifact each capacity is represented as a one-entry list, which preserves compatibility with earlier data handling while still yielding the scalar limit used by transfer and separation checks.


In [76]:
df_maximum_volume = pd.DataFrame([
    {"ModPlant": wb, "Maximum Volume": vol}
    for wb, vols in modplant_maximum_volume.items()
    for vol in vols
])

_ = tools.display_dataframe_to_user(name="ModPlant Maximum Volumes", dataframe=df_maximum_volume)

try:
    import gc
    del df_maximum_volume
    gc.collect()
except Exception:
    pass


**ModPlant Maximum Volumes**

,ModPlant,Maximum Volume
0,HC10,10
1,HC20,15
2,HC30,10
3,HC40,30


## ModPlant Initial Resources

The following table records the initial material allocation of the active plant configuration. These entries define the inventory component of the start state used by the planner.


In [77]:
df_resources = pd.DataFrame([
    {"ModPlant": wb, "Material": res[0], "Quantity": res[1]}
    for wb, res in modplant_resources.items()
])

_ = tools.display_dataframe_to_user(name="ModPlant Resources", dataframe=df_resources)

try:
    import gc
    del df_resources
    gc.collect()
except Exception:
    pass


**ModPlant Resources**

,ModPlant,Material,Quantity
0,HC10,A,10
1,HC20,B,10
2,HC30,C,10


## Initial State Construction and Display

This code builds an **immutable state snapshot** of all ModPlant, combining their initial contents and output port connections, then renders it in two tables.

* **`res_to_tuple(v)`**
  Normalizes resource data into a consistent tuple of strings:

  * Supports dicts (`{"material":…, "quantity":…, "unit":…}`)
  * Lists/tuples of dicts
  * Simple `(material, quantity)` pairs
  * Formats as `"Material : quantity unit"`

* **`build_initial_state(modplant_resources, modplant_interfaces)`**
  Creates the starting state as a sorted tuple of per-ModPlant records:

  * **Content** — current material(s) in the ModPlant
  * **Outputs** — all output interfaces initialized as unconnected (`""`)

* **`display_state(state_tuple)`**
  Generates two DataFrames:

  * **Initial ModPlant Contents** — which material and how much each ModPlant holds
  * **Initial ModPlant Ports (Connections)** — all output ports with current connections (empty at start)

* **`start_state`**
  The initial system state, used as input for BFS, optimization, or simulation routines.


In [78]:
def res_to_tuple(v):
    """Convert resource list to sorted tuple of strings"""
    if isinstance(v, dict):
        return (f"{v.get('material','')} : {float(v.get('quantity',0)):.1f} {v.get('unit','litre')}",)
    if isinstance(v, (list, tuple)) and v and isinstance(v[0], dict):
        return tuple(sorted(
            f"{d.get('material','')} : {float(d.get('quantity',0)):.1f} {d.get('unit','litre')}"
            for d in v
        ))
    if isinstance(v, (list, tuple)) and len(v) >= 2 and isinstance(v[0], str):
        return (f"{v[0]} : {float(v[1]):.1f} litre",)
    return ()

def build_initial_state(modplant_resources, modplant_interfaces):
    state = tuple(sorted(
        (
            wb,
            ("content", res_to_tuple(modplant_resources.get(wb, []))),
            ("outputs", tuple(sorted(
                (name, "", "") for t, name in ifaces if t == "Output"  # Modified: from (name, "") to (name, "", "")
            )))
        )
        for wb, ifaces in modplant_interfaces.items()
    ))
    return state

def display_state(state_tuple):
    """Display two DataFrames from the immutable state tuple."""

    df_state_contents = pd.DataFrame([
        {
            "ModPlant": wb,
            "Content": ", ".join(sorted(content_val))
        }
        for wb, (content_key, content_val), _ in state_tuple
    ])

    df_state_ports = pd.DataFrame([
        {
            "ModPlant": wb,
            "Port Type": "Output",
            "Port Name": port_name,
            "Connected To": target or "",
            "Material": material or ""  # Added Material column
        }
        for wb, _, (out_key, out_ports) in state_tuple
        for port_name, target, material in out_ports  # Modified: from port_name, target to port_name, target, material
    ])

    _ = tools.display_dataframe_to_user(name="Initial ModPlant Contents", dataframe=df_state_contents)
    _ = tools.display_dataframe_to_user(name="Initial ModPlant Ports (Connections)", dataframe=df_state_ports)

start_state = build_initial_state(modplant_resources, modplant_interfaces)
display_state(start_state)


**Initial ModPlant Contents**

,ModPlant,Content
0,HC10,A : 10.0 litre
1,HC20,B : 10.0 litre
2,HC30,C : 10.0 litre
3,HC40,


**Initial ModPlant Ports (Connections)**

,ModPlant,Port Type,Port Name,Connected To,Material
0,HC10,Output,HC10_Out1,,
1,HC10,Output,HC10_Out2,,
2,HC10,Output,HC10_Out3,,
3,HC20,Output,HC20_Out1,,
4,HC20,Output,HC20_Out2,,
5,HC20,Output,HC20_Out3,,
6,HC30,Output,HC30_Out1,,
7,HC40,Output,HC40_Out1,,
8,HC40,Output,HC40_Out2,,


## End State Construction

This function generates a **target state** that mirrors the structure of the initial state but enforces a uniform end condition.

* **`build_end_state_from_start(start_state_tuple)`**

  * Iterates over all ModPlant from the starting state.
  * Sets **content** to a fixed marker `("End",)` for every ModPlant.
  * Resets **outputs** to the same set of ports as in the start state, all left unconnected (`""`).
  * Returns a sorted, immutable tuple representing the canonical **end state**.

* **`end_state`**
  The result of applying the function to `start_state`.
  It represents the **goal configuration**: all ModPlant are marked finished, and no active connections remain.

* **`display_state(end_state)`**
  Shows two DataFrames:

  * **ModPlant Contents** — all entries display `End`.
  * **ModPlant Ports** — same ports as the start state, but all disconnected.

In [79]:
def build_end_state_from_start(start_state_tuple):
    """Create an end state where all ModPlant have 'End' and outputs are empty but identical."""
    end_state = tuple(sorted(
        (
            wb,
            ("content", ("End",)),
            ("outputs", tuple(sorted((port_name, "", "") for port_name, _, _ in outputs)))  # Modified here
        )
        for wb, (_, _), (_, outputs) in start_state_tuple
    ))
    return end_state

end_state = build_end_state_from_start(start_state)
display_state(end_state)


**Initial ModPlant Contents**

,ModPlant,Content
0,HC10,End
1,HC20,End
2,HC30,End
3,HC40,End


**Initial ModPlant Ports (Connections)**

,ModPlant,Port Type,Port Name,Connected To,Material
0,HC10,Output,HC10_Out1,,
1,HC10,Output,HC10_Out2,,
2,HC10,Output,HC10_Out3,,
3,HC20,Output,HC20_Out1,,
4,HC20,Output,HC20_Out2,,
5,HC20,Output,HC20_Out3,,
6,HC30,Output,HC30_Out1,,
7,HC40,Output,HC40_Out1,,
8,HC40,Output,HC40_Out2,,


## General Recipe Construction

This cell instantiates the order used for the reference case, derives the corresponding stage schedule via `build_schedule`, and stores the economic coefficients that later appear in the optimization objective. The default order reproduces the paper-aligned 6 L reference case; the seeded supplementary branch reuses `random_seed` when randomized plant generation is enabled.

The cell also defines `save_recipe_to_local`, which controls whether generated recipe artifacts are written into the repository-local `Recipe/XML` and `Recipe/Json` folders. The same switch can be overridden non-interactively through the environment variable `MODPLANT_SAVE_RECIPE_LOCAL`.


In [80]:
from pathlib import Path
from supporting_scripts.ModPlant_Flow_Generator import build_schedule
from supporting_scripts.ModPlant_Flow_To_General_Recipe import save_general_recipe_xml
import io
import os
import random
import sys

if hasattr(sys.stdout, "buffer"):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')

save_recipe_env = os.getenv('MODPLANT_SAVE_RECIPE_LOCAL')
save_recipe_to_local = True if save_recipe_env is None else save_recipe_env.strip().lower() in {'1', 'true', 'yes', 'on'}
local_recipe_root = Path('Recipe')
local_recipe_xml_dir = local_recipe_root / 'XML'
local_recipe_json_dir = local_recipe_root / 'Json'

use_original_order = use_original_modplants

order = {
  "volume": 6.0,
  "order": [
    "A",
    "B",
    "C",
    {"mix": {"rpm": 150, "duration": 30}}
  ],
  "ratio": {
    "A": [1],
    "B": [2],
    "C": [3]
  },
  "usage_and_settling": [3600, 300]
}

if not use_original_order:
    random.seed(random_seed)
    order_volume = 10

    letters = ['A', 'B', 'C']
    random.shuffle(letters)

    rpm_choices = list(range(50, 301, 50))
    def make_mix():
        return {"mix": {"rpm": random.choice(rpm_choices), "duration": random.randrange(100, 1001, 100)}}

    order_list = letters.copy()
    order_list.append(make_mix())

    parts = [random.randint(1, 8) for _ in letters]
    scale = 10 / sum(parts)
    ratio_vals = [max(1, round(v * scale)) for v in parts]
    adj = 10 - sum(ratio_vals)
    ratio_vals[0] += adj
    ratio = {k: [v] for k, v in zip(letters, ratio_vals)}

    usage_and_settling = [random.randrange(100, 5001, 100), random.randrange(100, 1001, 100)]

    order = {
      "volume": float(order_volume),
      "order": order_list,
      "ratio": ratio,
      "usage_and_settling": usage_and_settling
    }

order_profit_factor = (50 * order["volume"], -0.5)

print("Generating schedule from order.")
schedule = build_schedule(order)
print(f"save_recipe_to_local = {save_recipe_to_local}")


Generating schedule from order.
save_recipe_to_local = True


## Flowchart Rendering and General Recipe XML Export

The following cell renders the generated schedule as an ASCII flowchart and serializes it to an ISA-88 General Recipe XML file.

- If `save_recipe_to_local` is `True`, the XML is written into the repository-local `Recipe/XML` folder.
- If `save_recipe_to_local` is `False`, the supporting-script default output path is used instead.

The rendered schedule is intended as an inspection artifact for the companion notebook rather than as an execution backend in its own right.


In [81]:
from typing import Dict, List

def fmt_time(seconds: float) -> str:
    """Helper to format seconds into a readable string."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    return f"{seconds // 60:.0f}m {seconds % 60:.1f}s"

def _fmt_ratio_value(val) -> str:
    """Helper to format ratio values (handles lists or single numbers)."""
    if isinstance(val, list):
        return str(val[0])
    return str(val)

def make_box(title: str, lines: List[str]) -> List[str]:
    """Build a single ASCII box with a title + lines."""
    content = [title] + lines
    width = max((len(x) for x in content), default=0) + 2
    top = "┌" + "─" * width + "┐"
    bot = "└" + "─" * width + "┘"
    body = ["│ " + x.ljust(width - 2) + " │" for x in content]
    return [top] + body + [bot]

def _render_box_from_entry(entry: Dict) -> List[str]:
    """Create a box on the fly from one schedule entry."""
    t = entry["type"]
    title = entry["stage"]
    p = entry.get("params", {})
    dur = float(entry.get("duration_s", 0))

    lines = []
    if t == "dose":
        lines = [f"Portion: {p.get('portion_L', 0.0):.3f} L"]
    elif t == "mix":
        lines = [f"RPM: {p.get('rpm', 0)}", f"Duration: {fmt_time(dur)}"]
    elif t == "usage":
        lines = [f"Duration: {fmt_time(dur)}"]
    elif t == "collecting":
        lines = [f"Volume: {p.get('volume_L', 0.0):.3f} L"]
    elif t == "settling":
        lines = [f"Duration: {fmt_time(dur)}"]
    elif t == "sep":
        lines = [f"Volume: {p.get('volume_L', 0.0):.3f} L"]

    return make_box(title, lines)

def get_flowchart_text(schedule: List[Dict]) -> str:
    """
    Render a vertical ASCII flowchart and RETURN it as a string.
    This avoids output stream issues in multiprocess environments.
    """
    out: List[str] = []
    for i, e in enumerate(schedule):
        out.extend(_render_box_from_entry(e))
        if i < len(schedule) - 1:
            out.append(" " * 4 + "↓")
    return "\n".join(out)

print("\n=== ASCII FLOWCHART ===\n")
flowchart_output = get_flowchart_text(schedule)
print(flowchart_output, flush=True)

xml_output_dir = str(local_recipe_xml_dir) if save_recipe_to_local else None
xml_path = save_general_recipe_xml(schedule, order, out_dir=xml_output_dir)



=== ASCII FLOWCHART ===

┌──────────────────┐
│ Dosing A         │
│ Portion: 1.000 L │
└──────────────────┘
    ↓
┌──────────────────┐
│ Dosing B         │
│ Portion: 2.000 L │
└──────────────────┘
    ↓
┌──────────────────┐
│ Dosing C         │
│ Portion: 3.000 L │
└──────────────────┘
    ↓
┌─────────────────┐
│ Mix 150 rpm     │
│ RPM: 150        │
│ Duration: 30.0s │
└─────────────────┘
    ↓
┌────────────────────┐
│ Usage              │
│ Duration: 60m 0.0s │
└────────────────────┘
    ↓
┌───────────────────┐
│ Settling          │
│ Duration: 5m 0.0s │
└───────────────────┘
    ↓
┌─────────────────┐
│ Separation C    │
│ Volume: 3.000 L │
└─────────────────┘
    ↓
┌─────────────────┐
│ Separation B    │
│ Volume: 2.000 L │
└─────────────────┘
    ↓
┌─────────────────┐
│ Separation A    │
│ Volume: 1.000 L │
└─────────────────┘
[GeneralRecipe] Saved: Recipe/XML/GRecipe_A_B_C_Mix150_U_S_SepC_SepB_SepA.xml


## Transforming the General Recipe into Semantic Reaction Rules

The next cell converts the generated General Recipe XML into JSON and derives the semantic reaction-rule table used by the planner. Each row captures one process obligation with its required inputs, reaction family, reaction parameter, and result material label.

If `save_recipe_to_local` is `True`, the parsed JSON companion file is written into the repository-local `Recipe/Json` folder. Otherwise, the supporting-script default output path is used.


In [82]:
from supporting_scripts.ModPlant_General_Recipe_To_Json import save_parsed_recipe_json_by_id
from supporting_scripts.ModPlant_Reaction_Rules import generate_reaction_rules_from_general_recipe_json, rules_to_dataframe

json_output_dir = str(local_recipe_json_dir) if save_recipe_to_local else None
json_path = save_parsed_recipe_json_by_id(xml_path, out_dir=json_output_dir)

reaction_rules = generate_reaction_rules_from_general_recipe_json(json_path)
reaction_rules_df = rules_to_dataframe(reaction_rules)

_ = tools.display_dataframe_to_user(name="Semantic Reaction Rules", dataframe=reaction_rules_df)


[OK] Saved JSON to: Recipe/Json/GeneralRecipe_DA_DB_DC_M150_U_S_SepC_SepB_SepA_parsed.json


**Semantic Reaction Rules**

,Inputs,Reaction Type,Reaction Param,Result
0,,Dosing,A: 1.0 litre,A: 1.0 litre
1,A : 1.0 litre,Dosing,B: 2.0 litre,"A : 1.0 litre, B: 2.0 litre"
2,"A : 1.0 litre, B : 2.0 litre",Dosing,C: 3.0 litre,"A : 1.0 litre, B : 2.0 litre, C: 3.0 litre"
3,"A : 1.0 litre, B : 2.0 litre, C : 3.0 litre",Mix,150 rpm / 30 second,A_B_C_mixed : 6.0 litre
4,A_B_C_mixed : 6.0 litre,Usage,3600 second,A_B_C_used : 6.0 litre
5,A_B_C_used : 6.0 litre,Settling,300 second,A_B_C_settled : 6.0 litre
6,A_B_C_settled : 6.0 litre,Separation,C,A_B_settled : 3.0 litre
7,A_B_settled : 3.0 litre,Separation,B,A_settled : 1.0 litre
8,A_settled : 1.0 litre,Separation,A,End


## State and Transition Type Definitions

The following aliases document the immutable state representation used throughout the search. In particular, output ports carry both their connection target and an associated material label, and `CandidateTransition` denotes a pure, ID-free successor description before global graph integration.


## FSA-Style Legality Filtering

This notebook does not expose a separate automaton object named `FSA`. Instead, the legality layer described in the paper is realized as **state-dependent admissibility filtering** spread across the next blocks:

- port and connection feasibility checks in the connection helpers;
- capability and content checks in the concrete `compute_*_candidates` functions;
- symbolic-state resolution and rule dispatch in `check_rules_candidates(...)`; and
- the global stirring-capability precheck executed before full BFS expansion.

Taken together, these guards play the role of the paper's FSA: a candidate transition is emitted only if it is legal under the current material allocation, topology, capability set, and recipe progress.


In [83]:
from typing import Any, Dict, List, Optional, Set, Tuple, Union
from collections import deque
import concurrent.futures
import pandas as pd

ModPlantName = str
MaterialStr = str
ContentTuple = Tuple[str, Tuple[MaterialStr, ...]]
OutputsTuple = Tuple[str, Tuple[Tuple[str, str, str], ...]]
ModPlantStateTuple = Tuple[ModPlantName, ContentTuple, OutputsTuple]
FullState = Tuple[ModPlantStateTuple, ...]
Transition = Tuple[int, FullState, str, int, FullState, float, float]
CandidateTransition = Tuple[FullState, FullState, str, float, float]
SymbolicVars = Tuple[Tuple[str, float, float], ...]


## Symbolic Variable Utilities

This cell defines helper functions for detecting symbolic state metadata, parsing symbolic material expressions, and extracting the linear cost and duration coefficients that accompany deferred partial transfers.


In [84]:
def get_symbolic_vars(state: FullState) -> List[Tuple[str, float, float]]:
    if state and state[-1][0] == "__VARS__":
        content = state[-1][1][1]
        vars_list = []
        for s in content:
            parts = s.split(":")
            vars_list.append((parts[0], float(parts[1]), float(parts[2])))
        return vars_list
    return []

def remove_symbolic_vars(state: FullState) -> FullState:
    if state and state[-1][0] == "__VARS__":
        return state[:-1]
    return state

def parse_material_string_symbolic(material_str: str) -> Dict[str, Any]:
    """
    Returns dict: { 'Material': {'base': 10.0, 'vars': {'x': -1.0}} }
    """
    result = {}
    entries = material_str.split(',')
    for entry in entries:
        if ':' not in entry: continue
        name, qty_part = entry.split(':', 1)
        name = name.strip()
        qty_part = qty_part.strip()

        base_val = 0.0
        vars_dict = {}

        tokens = qty_part.split() # ['10.0', '+', '-1.0', 'x', 'litre'] or ['10.0', 'litre']

        try:
            if len(tokens) >= 5 and tokens[1] == '+' and tokens[3] in ['x', 'y', 'z']:
                base_val = float(tokens[0])
                coeff = float(tokens[2])
                var_name = tokens[3]
                vars_dict[var_name] = coeff
            else:
                base_val = float(tokens[0])
        except:
            continue

        result[name] = {'base': base_val, 'vars': vars_dict}
    return result

def get_symbolic_coeffs(state: FullState) -> Tuple[float, float]:
    """Extract the cost and duration coefficients (multipliers of x) from a symbolic state."""
    cost_c = 0.0
    dur_c = 0.0
    if state and state[-1][0] == "__VARS__":
        try:
            coeffs_tuple = None
            for item in state[-1]:
                if isinstance(item, tuple) and item[0] == "coeffs":
                    coeffs_tuple = item[1]
                    break

            if coeffs_tuple:
                for s in coeffs_tuple:
                    if s.startswith("cost:"):
                        cost_c = float(s.split(":")[1])
                    elif s.startswith("dur:"):
                        dur_c = float(s.split(":")[1])
        except:
            pass
    return cost_c, dur_c


## Material Parsing and Cost Utilities

The next cell defines the small pure helpers reused throughout candidate generation: operation-cost lookup, material-string parsing, total-volume accumulation, and tolerance-based material comparison.


In [85]:
import math
from typing import Dict

def get_operation_cost(
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    wb_name: str,
    operation: str
) -> int:
    """Extract cost (3rd value) of a given operation for a specific ModPlant."""
    for op_name, _, cost in modplant_ops.get(wb_name, []):
        if op_name == operation:
            return cost
    return 0

def parse_material_string(material_str: str) -> Dict[str, float]:
    """Parse 'A: 1.0 litre' -> {'A': 1.0}. Returns empty if symbolic."""
    result: Dict[str, float] = {}
    entries = material_str.split(',')
    for entry in entries:
        if ':' not in entry: continue
        name, qty = entry.split(':', 1)
        name = name.strip()
        try:
            amount = float(qty.strip().split()[0])
            result[name] = amount
        except ValueError:
            continue
    return result

def get_material_dict(contents: Tuple[str, ...]) -> Dict[str, float]:
    """Convert tuple of material strings to a material->amount dict."""
    return parse_material_string(', '.join(contents))

def parse_total_volume(content: Tuple[str, Tuple[str, ...]]) -> float:
    """Sum up all numeric quantities in a material content tuple."""
    total = 0.0
    for entry in content[1]:
        try:
            qty_str = entry.split(":")[1].strip().split(" ")[0]
            total += float(qty_str)
        except Exception:
            continue
    return total

def are_materials_equal(d1: Dict[str, float], d2: Dict[str, float], tol=1e-4) -> bool:
    """Performs a fuzzy match between two material dictionaries."""
    if set(d1.keys()) != set(d2.keys()):
        return False

    for k, v in d1.items():
        if abs(d2[k]) < 1e-9:
            continue

        if not math.isclose(v, d2[k], abs_tol=tol):
            return False

    return True


## Connection-State Helpers

This cell defines the pure connection logic used by the planner: port-availability checks, existing-connection lookup, immutable connection updates, and `pure_set_connection(...)`, which returns a connection candidate without mutating the global BFS structures. The same cell also provides the immutable draining and filling helpers used by subsequent transfer, dosing, and separation candidates.


In [86]:
def is_input_free(state: FullState, target_wb: str, target_port: str) -> bool:
    """Check if a given input port on target_wb is free (not referenced by any output)."""
    for _, _, (_, outputs) in state:
        for _, connected_to, _ in outputs:
            if connected_to == target_port:
                return False
    return True

def get_free_outputs(state: FullState, wb_name: str) -> List[str]:
    """Return list of free (unconnected) output ports of a ModPlant."""
    for wb, _, (_, outputs) in state:
        if wb == wb_name:
            return [port_name for port_name, connected, material in outputs if not connected]
    return []

def get_all_input_ports(
    wb_name: str,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]]
) -> List[str]:
    """Return all input port names for a given ModPlant based on interface definitions."""
    return [
        port_name
        for port_type, port_name in modplant_interfaces.get(wb_name, [])
        if port_type == "Input"
    ]

def get_connected_port_pair(
    state: FullState,
    from_wb: str,
    to_wb: str,
    material: str = ""
) -> Tuple[str, str]:
    """
    Find an existing connection from 'from_wb' to 'to_wb'.
    Returns (out_port, in_port), optionally filtered by material.
    """
    for wb, _, (_, outputs) in state:
        if wb == from_wb:
            for port_name, conn, mat in outputs:
                if conn.startswith(to_wb + ":") and (not material or mat == material):
                    return port_name, conn.split(":")[1]
    return "", ""

def get_free_output_port(state: FullState, wb: str) -> str:
    """Return a single free output port of 'wb', or empty string if none is free."""
    for wb_name, _, (_, outputs) in state:
        if wb_name == wb:
            for port_name, conn, material in outputs:
                if not conn:
                    return port_name
    return ""

def get_free_input_port(
    wb_name: str,
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]]
) -> str:
    """Return a single free input port of 'wb_name', or empty string if none is free."""
    used_ports = set()
    for _, _, (_, outputs) in state:
        for _, conn, material in outputs:
            if conn.startswith(wb_name + ":"):
                used_ports.add(conn.split(":")[1])
    return next(
        (
            port
            for t, port in modplant_interfaces[wb_name]
            if t == "Input" and port not in used_ports
        ),
        ""
    )

def update_connection(
    state: FullState,
    wb_from: str,
    out_port: str,
    wb_to: str,
    in_port: str,
    material: str
) -> FullState:
    """Return a new state where wb_from:out_port is connected to wb_to:in_port for material."""
    new_state: List[ModPlantStateTuple] = []
    target_str = wb_to + ":" + in_port
    for wb, (content_key, content_val), (out_key, outputs) in state:
        if wb != wb_from:
            new_state.append((wb, (content_key, content_val), (out_key, outputs)))
        else:
            new_outputs = tuple(
                (name, target_str, material) if name == out_port else (name, conn, mat)
                for name, conn, mat in outputs
            )
            new_state.append((wb, (content_key, content_val), (out_key, new_outputs)))
    return tuple(sorted(new_state))

def pure_set_connection(
    state: FullState,
    wb_from: str,
    out_port: str,
    wb_to: str,
    in_port: str,
    material: str,
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
) -> Optional[Tuple[FullState, CandidateTransition]]:
    """
    Pure version of 'set_connection':
    - does NOT touch visited / queue / transition_list
    - returns (new_state, candidate_transition) or None if no connection possible
    """
    if not is_input_free(state, wb_to, in_port):
        return None

    new_state = update_connection(state, wb_from, out_port, wb_to, in_port, material)
    op_str = f"Connect({out_port} -> {in_port}) for {material}"

    cost1 = get_operation_cost(modplant_ops, wb_from, "Connect")
    cost2 = get_operation_cost(modplant_ops, wb_to, "Connect")
    cost = float(cost1 + cost2)
    duration = 3.0

    cand: CandidateTransition = (state, new_state, op_str, cost, duration)
    return new_state, cand

def apply_draining(state: FullState, wb: str) -> FullState:
    """Return a new state where all material is removed from ModPlant 'wb'."""
    new_state: List[ModPlantStateTuple] = []
    for w, (content_key, content_val), (out_key, outputs) in state:
        if w == wb:
            new_state.append((w, (content_key, ()), (out_key, outputs)))
        else:
            new_state.append((w, (content_key, content_val), (out_key, outputs)))
    return tuple(sorted(new_state))

def apply_filling(state: FullState, wb: str, new_content: Tuple[str, ...]) -> FullState:
    """Return a new state where ModPlant 'wb' content is replaced by 'new_content'."""
    new_state: List[ModPlantStateTuple] = []
    for w, (content_key, content_val), (out_key, outputs) in state:
        if w == wb:
            new_state.append((w, (content_key, new_content), (out_key, outputs)))
        else:
            new_state.append((w, (content_key, content_val), (out_key, outputs)))
    return tuple(sorted(new_state))

def apply_partial_draining(
    state: FullState,
    wb: str,
    to_drain: Dict[str, float]
) -> FullState:
    """Return a new state where specified material quantities are removed from ModPlant 'wb'."""
    new_state: List[ModPlantStateTuple] = []
    for w, (content_key, content_val), (out_key, outputs) in state:
        if w != wb:
            new_state.append((w, (content_key, content_val), (out_key, outputs)))
            continue

        current_dict = get_material_dict(content_val)
        for k, v in to_drain.items():
            if k in current_dict:
                current_dict[k] -= v
                if current_dict[k] <= 0:
                    del current_dict[k]
        new_content = tuple(f"{k}: {v} litre" for k, v in current_dict.items())
        new_state.append((w, (content_key, new_content), (out_key, outputs)))
    return tuple(sorted(new_state))

def apply_partial_filling(
    state: FullState,
    wb: str,
    to_fill: Dict[str, float]
) -> FullState:
    """Return a new state where specified material quantities are added to ModPlant 'wb'."""
    new_state: List[ModPlantStateTuple] = []
    for w, (content_key, content_val), (out_key, outputs) in state:
        if w != wb:
            new_state.append((w, (content_key, content_val), (out_key, outputs)))
            continue

        current_dict = get_material_dict(content_val)
        for k, v in to_fill.items():
            current_dict[k] = current_dict.get(k, 0) + v
        new_content = tuple(f"{k}: {v} litre" for k, v in current_dict.items())
        new_state.append((w, (content_key, new_content), (out_key, outputs)))
    return tuple(sorted(new_state))


## Partial-Transfer Candidate Generator

The following cell implements the transfer branching logic used during BFS. With `use_lazy_transfer_states = True`, it produces symbolic successor states with deferred variables and stores the corresponding cost and duration coefficients for later resolution. With `use_lazy_transfer_states = False`, it enumerates concrete transfer quantities at multiples of `min_transfer_volume` and additionally includes the full admissible transfer amount.


In [87]:
def enumerate_concrete_transfer_amounts(max_transfer: float, step: float) -> List[float]:
    if max_transfer <= 1e-6:
        return []

    amounts: List[float] = []
    multiplier = 1
    while True:
        candidate = multiplier * step
        if candidate < max_transfer - 1e-9:
            amounts.append(candidate)
            multiplier += 1
            continue
        break

    amounts.append(max_transfer)

    deduped: List[float] = []
    for amount in amounts:
        if amount <= 1e-6:
            continue
        if any(abs(amount - existing) <= 1e-9 for existing in deduped):
            continue
        deduped.append(amount)
    return deduped

def check_transfer_part_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
) -> List[CandidateTransition]:
    results: List[CandidateTransition] = []

    if get_symbolic_vars(state):
        return []

    for wb1, (content_key1, contents1), (_, _) in state:
        if wb1 == "__VARS__": continue
        mat_dict1 = get_material_dict(contents1)
        if not mat_dict1: continue

        material_name = next(iter(mat_dict1.keys()))
        available_amount = mat_dict1.get(material_name, 0.0)

        if available_amount <= 1e-6: continue
        if not any(op[0] == "Draining" for op in modplant_ops.get(wb1, [])): continue

        for wb2, (content_key2, contents2), (_, _) in state:
            if wb1 == wb2 or wb2 == "__VARS__": continue

            mat_dict2 = get_material_dict(contents2)
            if mat_dict2 and (len(mat_dict2) != 1 or material_name not in mat_dict2):
                continue

            max_capacity = modplant_maximum_volume.get(wb2, [0.0])[0]
            current_vol2 = sum(mat_dict2.values())
            space_available = max_capacity - current_vol2

            if space_available <= 1e-6: continue
            if not any(op[0] == "Filling" for op in modplant_ops.get(wb2, [])): continue

            out_port, in_port = get_connected_port_pair(state, wb1, wb2, material_name)
            local_state = state
            local_cands = []

            if not out_port or not in_port:
                out_port = get_free_output_port(state, wb1)
                in_port = get_free_input_port(wb2, state, modplant_interfaces)
                if not out_port or not in_port: continue

                conn_result = pure_set_connection(
                    state, wb1, out_port, wb2, in_port, material_name, modplant_ops
                )
                if conn_result is None: continue
                local_state, connect_cand = conn_result
                local_cands.append(connect_cand)

            drain_cost_unit = next((float(op[2]) for op in modplant_ops.get(wb1, []) if op[0] == "Draining"), 0.0)
            fill_cost_unit = next((float(op[2]) for op in modplant_ops.get(wb2, []) if op[0] == "Filling"), 0.0)
            cost_coeff = drain_cost_unit + fill_cost_unit

            drain_speed = next((float(op[1]) for op in modplant_ops.get(wb1, []) if op[0] == "Draining"), 0.01)
            fill_speed = next((float(op[1]) for op in modplant_ops.get(wb2, []) if op[0] == "Filling"), 0.01)
            transfer_speed = min(drain_speed, fill_speed)

            dur_coeff = 1.0 / transfer_speed if transfer_speed > 1e-9 else 0.0
            max_transfer = min(available_amount, space_available)

            if use_lazy_transfer_states:
                var_name = "x"
                new_state_list = []
                for w, (c_key, c_val), (o_key, outputs) in local_state:
                    if w == wb1:
                        new_c_list = []
                        for k, v in mat_dict1.items():
                            if k == material_name:
                                new_c_list.append(f"{k}: {v} + -1.0 {var_name} litre")
                            else:
                                new_c_list.append(f"{k}: {v} litre")
                        new_state_list.append((w, (c_key, tuple(sorted(new_c_list))), (o_key, outputs)))
                    elif w == wb2:
                        base_v = mat_dict2.get(material_name, 0.0)
                        new_c = (f"{material_name}: {base_v} + 1.0 {var_name} litre",)
                        new_state_list.append((w, (c_key, new_c), (o_key, outputs)))
                    else:
                        new_state_list.append((w, (c_key, c_val), (o_key, outputs)))

                constraint_entry = (
                    "__VARS__",
                    ("vars", (f"{var_name}:0.0:{max_transfer}",)),
                    ("coeffs", (f"cost:{cost_coeff}", f"dur:{dur_coeff}"))
                )
                new_state_list.append(constraint_entry)
                symbolic_state = tuple(sorted(new_state_list, key=lambda x: x[0]))

                op_str = f"Variable Transfer setup ({wb1}->{wb2}) for {material_name}: {var_name} in [0, {max_transfer:.2f}]"
                local_cands.append((local_state, symbolic_state, op_str, 0.0, 0.0))
            else:
                transfer_amounts = enumerate_concrete_transfer_amounts(max_transfer, min_transfer_volume)
                for transfer_amount in transfer_amounts:
                    transfer_dict = {material_name: transfer_amount}
                    drained_state = apply_partial_draining(local_state, wb1, transfer_dict)
                    filled_state = apply_partial_filling(drained_state, wb2, transfer_dict)
                    op_str = (
                        f"Dosing: Open Valve of {out_port} only, "
                        f"Draining({wb1}), Filling({wb2}), ({material_name}: {transfer_amount:.2f} litre)"
                    )
                    local_cands.append((local_state, filled_state, op_str, cost_coeff * transfer_amount, dur_coeff * transfer_amount))

            results.extend(local_cands)

    return results


## Concrete Dosing Candidate Generation

When a reaction-rule row prescribes a concrete dose, the following function enumerates admissible source-target pairs, reuses or creates the required connection, applies partial draining and filling immutably, and attaches cost and duration values derived from the active draining and filling rates.


In [88]:
def compute_dosing_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    """Generate pure Dosing candidates for a single state and rule."""
    results: List[CandidateTransition] = []

    req_dose = parse_material_string(rule_row["Reaction Param"])
    req_inputs = parse_material_string(rule_row["Inputs"]) if rule_row["Inputs"] else None

    for wb1, (content_key1, contents1), (_, _) in state:
        if not any(op[0] == "Draining" for op in modplant_ops.get(wb1, [])):
            continue
        wb1_dict = get_material_dict(contents1)
        if any(wb1_dict.get(k, 0) < v for k, v in req_dose.items()):
            continue

        material_name = ", ".join(req_dose.keys()) if req_dose else ""

        for wb2, (content_key2, contents2), (_, _) in state:
            if wb1 == wb2:
                continue
            if not any(op[0] == "Filling" for op in modplant_ops.get(wb2, [])):
                continue
            wb2_dict = get_material_dict(contents2)

            if req_inputs:
                if wb2_dict != req_inputs:
                    continue
            else:
                if wb2_dict:
                    continue

            out_port, in_port = get_connected_port_pair(state, wb1, wb2, material_name)
            local_state = state
            local_cands: List[CandidateTransition] = []

            if not out_port or not in_port:
                out_port = get_free_output_port(state, wb1)
                in_port = get_free_input_port(wb2, state, modplant_interfaces)
                if not out_port or not in_port:
                    continue
                conn_result = pure_set_connection(
                    state, wb1, out_port, wb2, in_port, material_name, modplant_ops
                )
                if conn_result is None:
                    continue
                local_state, connect_cand = conn_result
                local_cands.append(connect_cand)

            drained = apply_partial_draining(local_state, wb1, req_dose)
            filled = apply_partial_filling(drained, wb2, req_dose)

            dose_amount = sum(req_dose.values())
            draining_speed = float(next(
                (op[1] for op in modplant_ops[wb1] if op[0] == "Draining"),
                1.0
            ))
            filling_speed = float(next(
                (op[1] for op in modplant_ops[wb2] if op[0] == "Filling"),
                1.0
            ))
            speed = min(draining_speed, filling_speed) if draining_speed and filling_speed else 1.0
            duration = dose_amount / speed if speed > 0 else 0.0

            cost_draining = next((op[2] for op in modplant_ops[wb1] if op[0] == "Draining"), 0) * dose_amount
            cost_filling = next((op[2] for op in modplant_ops[wb2] if op[0] == "Filling"), 0) * dose_amount
            cost = float(cost_draining + cost_filling)

            raw_param = rule_row['Reaction Param']
            formatted_parts = []
            if raw_param:
                for p in raw_param.split(','):
                    if ':' in p:
                        name, rest = p.split(':', 1)
                        tokens = rest.strip().split()
                        if tokens:
                            try:
                                val = float(tokens[0])
                                unit = " ".join(tokens[1:])
                                formatted_parts.append(f"{name.strip()}: {val:.2f} {unit}")
                            except ValueError:
                                formatted_parts.append(p.strip())
                        else:
                            formatted_parts.append(p.strip())
                    else:
                        formatted_parts.append(p.strip())
                formatted_param = ", ".join(formatted_parts)
            else:
                formatted_param = ""

            op_str = (
                f"Dosing: Open Valve of {out_port} only, "
                f"Draining({wb1}), Filling({wb2}), ({formatted_param})"
            )

            local_cands.append((local_state, filled, op_str, cost, duration))
            results.extend(local_cands)

    return results


## Mixing Result Formatting Helper

This small helper normalizes reaction results so that downstream mixing candidates retain an explicit material label together with the preserved total volume.


In [89]:
def format_result_content(result_mat_str: str, total_vol: float, unit: str = "litre") -> str:
    """
    Format the result material string with total volume.
    Returns a string in the format "Material : volume unit"
    """
    if ":" in result_mat_str:
        return result_mat_str
    else:
        return f"{result_mat_str} : {total_vol:.1f} {unit}"


## Mixing Candidate Generator

The following function matches recipe-side mixing requirements against the current vessel contents and module-specific stirring capabilities. Only modules with the required input multiset and an admissible stirring parameter generate successor states.


In [90]:
def compute_mixing_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    results: List[CandidateTransition] = []

    req_inputs = parse_material_string(rule_row["Inputs"])
    if not req_inputs: return []

    param_str = str(rule_row["Reaction Param"])
    if "/" in param_str:
        parts = param_str.split('/')
        rpm_str = parts[0].strip().split()[0]
        dur_str = parts[1].strip().split()[0]
    else:
        rpm_str = "0"
        dur_str = param_str.strip().split()[0] if param_str.strip() else "0"

    try:
        duration = float(dur_str)
    except:
        duration = 0.0

    for wb, (content_key, contents), (_, _) in state:
        content_dict = get_material_dict(contents)

        if not are_materials_equal(content_dict, req_inputs, tol=1e-4): continue

        matching_mix = next(
            (op for op in modplant_ops.get(wb, []) if op[0] == "Stirring" and str(op[1]) == rpm_str),
            None
        )
        if not matching_mix: continue

        cost = float(matching_mix[2])

        total_vol = sum(content_dict.values())
        unit = "litre"
        if contents:
            parts = contents[0].split()
            if parts: unit = parts[-1]

        result_mat = rule_row["Result"]
        new_content_str = format_result_content(result_mat, total_vol, unit)

        new_state_list = []
        for w, (c_key, c_val), (out_key, outputs) in state:
            if w == wb:
                new_state_list.append((w, (c_key, (new_content_str,)), (out_key, outputs)))
            else:
                new_state_list.append((w, (c_key, c_val), (out_key, outputs)))
        new_state = tuple(sorted(new_state_list))

        op_str = f"Stirring ({wb}), {rpm_str}rpm for {duration}s"
        results.append((state, new_state, op_str, cost, duration))
    return results


## Usage Candidate Generator

This cell implements usage transitions. It matches the required intermediate material, relabels the consumed content to the recipe result while preserving total volume, and currently records zero transition duration in the returned edge even though the recipe parameter is parsed and reflected in the operation string.


In [91]:
def compute_usage_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    """Generate pure Usage candidates for a single state and rule."""

    results: List[CandidateTransition] = []

    req_inputs = parse_material_string(rule_row["Inputs"])

    try:
        duration_str = rule_row["Reaction Param"].strip().split()[0]
        duration_val = float(duration_str)
    except Exception:
        return results

    for wb, (content_key, contents), (_, _) in state:
        content_dict = get_material_dict(contents)

        if not are_materials_equal(content_dict, req_inputs): continue

        has_none_op = any(op[0] == "None" for op in modplant_ops.get(wb, []))
        if not has_none_op:
            continue

        result_mat = rule_row["Result"]

        total_vol = sum(content_dict.values())

        unit = "litre"
        if contents:
            parts = contents[0].split()
            if parts: unit = parts[-1]

        new_content_str = f"{result_mat} : {total_vol:.2f} {unit}"
        new_content = (new_content_str,)

        new_state_list: List[ModPlantStateTuple] = []
        for w, (c_key, c_val), (out_key, outputs) in state:
            if w == wb:
                new_state_list.append((w, (c_key, new_content), (out_key, outputs)))
            else:
                new_state_list.append((w, (c_key, c_val), (out_key, outputs)))
        new_state = tuple(sorted(new_state_list))

        op_str = f"Usage ({wb}), {duration_val}s: None"
        results.append((state, new_state, op_str, 0.0, 0.0))

    return results


## Settling Candidate Generator

Settling candidates require exact content matching and a supported `Settling` capability. The operation string carries the recipe duration, while the returned transition currently keeps duration `0.0` to preserve the semantics of the public artifact.


In [92]:
def compute_settling_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    """Generate pure Settling candidates for a single state and rule."""
    results: List[CandidateTransition] = []

    req_inputs = parse_material_string(rule_row["Inputs"])

    try:
        duration_str = rule_row["Reaction Param"].strip().split()[0]
        duration_val = float(duration_str)
    except Exception:
        return results

    for wb, (content_key, contents), (_, _) in state:
        content_dict = get_material_dict(contents)
        if not are_materials_equal(content_dict, req_inputs): continue

        settling_op = next((op for op in modplant_ops.get(wb, []) if op[0] == "Settling"), None)
        if not settling_op:
            continue

        cost = float(settling_op[2])
        result_mat = rule_row["Result"]

        new_state_list: List[ModPlantStateTuple] = []
        for w, (c_key, c_val), (out_key, outputs) in state:
            if w == wb:
                new_content = (f"{result_mat}",)
                new_state_list.append((w, (c_key, new_content), (out_key, outputs)))
            else:
                new_state_list.append((w, (c_key, c_val), (out_key, outputs)))
        new_state = tuple(sorted(new_state_list))

        op_str = f"Settling ({wb}), {duration_val}s: Settling"
        results.append((state, new_state, op_str, cost, 0.0))

    return results


## Separation Candidate Generator

Separation candidates move one named component from a source module to a compatible receiver, update the source composition to the rule result, and optionally terminate directly in `end_state`. As in the current artifact, the transfer duration is computed internally but the emitted transition duration remains `0.0`.


In [93]:
def compute_separation_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    rule_row: pd.Series,
) -> List[CandidateTransition]:
    """Generate pure Separation candidates for a single state and rule."""
    results: List[CandidateTransition] = []

    req_inputs = parse_material_string(rule_row["Inputs"])
    result_material = parse_material_string(rule_row["Result"])
    separated_comp = rule_row["Reaction Param"].strip()

    for wb1, (content_key1, contents1), (_, _) in state:
        content_dict1 = get_material_dict(contents1)
        if content_dict1 != req_inputs:
            continue
        if not any(op[0] == "Draining" for op in modplant_ops.get(wb1, [])):
            continue

        for wb2, (content_key2, contents2), (_, _) in state:
            if wb1 == wb2:
                continue
            content_dict2 = get_material_dict(contents2)
            if content_dict2 and (list(content_dict2.keys()) != [separated_comp]):
                continue

            capacity_left = float('inf')
            for op in modplant_ops.get(wb2, []):
                if op[0] == "Filling":
                    capacity_left = modplant_maximum_volume.get(wb2, [0])[0]
                    break
            used_volume = sum(content_dict2.values())
            remaining_capacity = capacity_left - used_volume
            if remaining_capacity <= 0:
                continue

            source_volume = sum(req_inputs.values())
            target_volume = (
                0.0 if rule_row["Result"] == "End" else sum(result_material.values())
            )
            transfer_volume = max(0.0, source_volume - target_volume)
            if transfer_volume > remaining_capacity:
                continue

            out_port, in_port = get_connected_port_pair(state, wb1, wb2, separated_comp)
            local_state = state
            local_cands: List[CandidateTransition] = []

            if not out_port or not in_port:
                out_port = get_free_output_port(state, wb1)
                in_port = get_free_input_port(wb2, state, modplant_interfaces)
                if not out_port or not in_port:
                    continue
                conn_result = pure_set_connection(
                    state, wb1, out_port, wb2, in_port, separated_comp, modplant_ops
                )
                if conn_result is None:
                    continue
                local_state, connect_cand = conn_result
                local_cands.append(connect_cand)

            drain_dict = {separated_comp: transfer_volume}
            drained = apply_partial_draining(local_state, wb1, drain_dict)
            filled = apply_partial_filling(drained, wb2, drain_dict)

            draining_speed = float(next(
                (op[1] for op in modplant_ops[wb1] if op[0] == "Draining"),
                1.0
            ))
            filling_speed = float(next(
                (op[1] for op in modplant_ops[wb2] if op[0] == "Filling"),
                1.0
            ))
            speed = min(draining_speed, filling_speed)
            duration_val = transfer_volume / speed if speed > 0 else 0.0

            cost_draining = next((op[2] for op in modplant_ops[wb1] if op[0] == "Draining"), 0)
            cost_filling = next((op[2] for op in modplant_ops[wb2] if op[0] == "Filling"), 0)
            cost = float(cost_draining + cost_filling)

            final_state_list: List[ModPlantStateTuple] = []
            for w, (c_key, c_val), (out_key, outputs) in filled:
                if w == wb1:
                    new_content = tuple(f"{k}: {v} litre" for k, v in result_material.items())
                    final_state_list.append((w, (c_key, new_content), (out_key, outputs)))
                else:
                    final_state_list.append((w, (c_key, c_val), (out_key, outputs)))
            final_state = tuple(sorted(final_state_list))

            op_str = (
                f"Separation: Open Valve of {out_port} only, "
                f"Draining({wb1}), Filling({wb2}), "
                f"({separated_comp}: {transfer_volume:.2f} litre)"
            )

            if rule_row["Result"] == "End":
                local_cands.append((local_state, end_state, "End, " + op_str, cost, 0.0))
            else:
                local_cands.append((local_state, final_state, op_str, cost, 0.0))

            results.extend(local_cands)

    return results


## Lazy Instantiation of Symbolic Transfer States

When `use_lazy_transfer_states = True`, the following function operationalizes deferred-variable resolution. Given a symbolic state and a target material requirement, it solves the pending transfer variable, checks feasibility against the stored bounds, and materializes a concrete state only when a downstream rule requires a committed quantity. In non-lazy mode this function remains available but is not used, because transfer quantities are materialized eagerly and `min_transfer_volume` applies only to that concrete enumeration branch.


In [94]:
def solve_and_instantiate(
    state: FullState,
    wb_name: str,
    required_material: Dict[str, float]
) -> Optional[Tuple[FullState, float]]:
    """
    Resolve symbolic variable x for a specific ModPlant and instantiate a concrete state.
    No persistent cache to keep runs stateless across kernels.
    """
    vars_list = get_symbolic_vars(state)
    if not vars_list:
        return None

    var_name, v_min, v_max = vars_list[0]

    target_content = None
    for w, (_, content), _ in state:
        if w == wb_name:
            target_content = content
            break
    if target_content is None:
        return None

    solved_x = None

    if not required_material:
        for entry in target_content:
            if var_name not in entry:
                continue
            if "_mixed" in entry:
                continue
            try:
                rhs = entry.split(":", 1)[1]
                tokens = rhs.split()
                if var_name in tokens:
                    idx = tokens.index(var_name)
                    if idx > 0 and idx + 1 < len(tokens):
                        coeff = float(tokens[idx - 1])
                        base = float(tokens[0])
                        if abs(coeff) > 1e-9:
                            solved_x = -base / coeff
                            break
            except Exception:
                continue

    else:
        current_sym = parse_material_string_symbolic(', '.join(target_content))
        for mat, req_qty in required_material.items():
            if mat not in current_sym:
                return None
            info = current_sym[mat]
            base = info['base']
            coeffs = info['vars']

            if var_name not in coeffs:
                if abs(base - req_qty) > 1e-6:
                    return None
            else:
                coeff = coeffs[var_name]
                if abs(coeff) < 1e-9:
                    if abs(base - req_qty) > 1e-6:
                        return None
                else:
                    val = (req_qty - base) / coeff
                    if solved_x is not None and abs(solved_x - val) > 1e-6:
                        return None
                    solved_x = val

    if solved_x is None:
        return None

    if not (v_min - 1e-6 <= solved_x <= v_max + 1e-6):
        return None
    if solved_x < v_min:
        solved_x = v_min
    if solved_x > v_max:
        solved_x = v_max

    new_state_list = []
    for w, (c_key, c_val), (o_key, outputs) in state:
        if w == "__VARS__":
            continue

        has_var = any(var_name in line for line in c_val)
        if not has_var:
            new_state_list.append((w, (c_key, c_val), (o_key, outputs)))
            continue

        parsed = parse_material_string_symbolic(', '.join(c_val))
        new_mat_strs = []
        for m, info in parsed.items():
            qty = info['base']
            if var_name in info['vars']:
                qty += info['vars'][var_name] * solved_x
            if abs(qty) < 1e-6:
                continue
            new_mat_strs.append(f"{m}: {qty:.2f} litre")

        new_state_list.append((w, (c_key, tuple(sorted(new_mat_strs))), (o_key, outputs)))

    concrete_state = tuple(sorted(new_state_list, key=lambda x: x[0]))
    return concrete_state, solved_x


## Reaction Rule Candidate Dispatcher

This cell combines the symbolic and concrete branches of rule evaluation. For concrete states it dispatches to the corresponding `compute_*_candidates` function. For symbolic states it first attempts lazy instantiation and then emits the corresponding `Resolve(...)` transition before further concrete expansion.


In [95]:
def check_rules_candidates(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    reaction_rules_df: pd.DataFrame,
) -> List[CandidateTransition]:
    results: List[CandidateTransition] = []
    is_symbolic = bool(get_symbolic_vars(state))

    for _, rule_row in reaction_rules_df.iterrows():
        raw_inputs = rule_row["Inputs"]
        req_inputs = parse_material_string(raw_inputs)
        rtype = rule_row["Reaction Type"]

        rpm_required = None
        if rtype == "Mix":
            param_str = str(rule_row["Reaction Param"])
            if "/" in param_str:
                rpm_required = param_str.split('/')[0].strip().split()[0]
            else:
                tokens = param_str.strip().split()
                rpm_required = tokens[0] if tokens else None

        if is_symbolic:
            cost_coeff, dur_coeff = get_symbolic_coeffs(state)

            for wb, _, _ in state:
                if wb == "__VARS__": continue

                if rpm_required is not None:
                    ops = modplant_ops.get(wb, [])
                    if not any(op[0] == "Stirring" and str(op[1]) == rpm_required for op in ops):
                        continue

                sol = solve_and_instantiate(state, wb, req_inputs)
                if sol:
                    concrete_state, solved_x = sol

                    resolve_cost = cost_coeff * solved_x
                    resolve_dur = dur_coeff * solved_x

                    mat_name = list(req_inputs.keys())[0] if req_inputs else "Unknown"
                    op_str = f"Resolve(x={solved_x:.2f}) for {mat_name}"

                    results.append((state, concrete_state, op_str, resolve_cost, resolve_dur))
            continue

        candidates = []

        if rtype == "Dosing":
            candidates = compute_dosing_candidates(state, modplant_interfaces, modplant_ops, rule_row)
        elif rtype == "Mix":
            candidates = compute_mixing_candidates(state, modplant_interfaces, modplant_ops, rule_row)
        elif rtype == "Usage":
            candidates = compute_usage_candidates(state, modplant_interfaces, modplant_ops, rule_row)
        elif rtype == "Settling":
            candidates = compute_settling_candidates(state, modplant_interfaces, modplant_ops, rule_row)
        elif rtype == "Separation":
            candidates = compute_separation_candidates(state, modplant_interfaces, modplant_ops, rule_row)

        for (cand_from_state, to_state, op_str, op_cost, op_dur) in candidates:
            has_conn_change = False
            conn_op_str = ""
            conn_cost = 0.0
            conn_dur = 0.0

            if len(state) == len(cand_from_state):
                 for i in range(len(state)):
                    w1, _, out1 = state[i]
                    w2, _, out2 = cand_from_state[i]
                    if w1 == "__VARS__" or w2 == "__VARS__": continue

                    if out1 != out2:
                        has_conn_change = True
                        for p_idx, port_def in enumerate(out2[1]):
                            old_def = out1[1][p_idx]
                            if port_def != old_def and port_def[1] != "":
                                target_full = port_def[1]
                                target_clean = target_full.split(":")[1] if ":" in target_full else target_full
                                conn_op_str += f"Connect({port_def[0]} -> {target_clean}) for {port_def[2]} & "
                                conn_cost += 2.0
                                conn_dur += 3.0

            if has_conn_change:
                conn_op_str = conn_op_str.rstrip(" & ")
                results.append((state, cand_from_state, conn_op_str, conn_cost, conn_dur))
            else:
                results.append((state, to_state, op_str, op_cost, op_dur))

    return results


## Parallel BFS Reachable-State Graph Construction

The next cell defines `expand_state_pure(...)` and `run_bfs(...)`, the threaded BFS implementation used to construct the **reachable-state graph**. Candidate generation stays pure at the worker level; only the main thread merges visited states and transitions. The same cell performs the paper-facing stirring-capability precheck, builds the transition table for the reachable graph, and falls back to an explicit infeasible stub sequence when no end-reaching transition is found.


In [96]:
def expand_state_pure(
    state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    modplant_ops: Dict[str, List[Tuple[str, Union[str, float], int]]],
    reaction_rules_df: pd.DataFrame,
) -> List[CandidateTransition]:
    """
    Pure expansion of a single state:
    - Transfer operations
    - Reaction Rules (Dosing / Mix / Usage / Settling / Separation)
    """
    candidates: List[CandidateTransition] = []
    candidates.extend(check_transfer_part_candidates(state, modplant_interfaces, modplant_ops))
    candidates.extend(check_rules_candidates(state, modplant_interfaces, modplant_ops, reaction_rules_df))
    return candidates

def run_bfs(
    start_state: FullState,
    modplant_interfaces: Dict[str, List[Tuple[str, str]]],
    max_steps: Optional[int] = None,
    num_threads: int = 8,
    batch_size: Optional[int] = None,
) -> Tuple[List[Transition], List[FullState]]:
    """
    Multi-threaded BFS:
    - Uses a thread pool to expand states via expand_state_pure (pure function).
    - Only the main thread modifies visited / state_list / queue / transition_list.
    - max_steps: if not None, limit BFS depth; otherwise, run without a depth limit.
    """
    if batch_size is None:
        batch_size = num_threads

    visited: Dict[FullState, int] = {}
    state_list: List[FullState] = []
    transition_list: List[Transition] = []
    existing_transitions: Set[Tuple[int, str, int, FullState]] = set()
    queue: deque = deque()

    start_id = 0
    visited[start_state] = start_id
    state_list.append(start_state)
    queue.append((start_state, start_id, 0))

    end_id = 1
    visited[end_state] = end_id
    state_list.append(end_state)
    queue.append((end_state, end_id, 0))

    def add_transition(
        from_id: int,
        from_state: FullState,
        op_str: str,
        to_id: int,
        to_state: FullState,
        cost: float,
        duration: float,
    ) -> None:
        key = (from_id, op_str, to_id, to_state)
        if key in existing_transitions:
            return
        transition_list.append((from_id, from_state, op_str, to_id, to_state, cost, duration))
        existing_transitions.add(key)

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        while queue:
            batch: List[Tuple[FullState, int, int]] = []

            while queue and len(batch) < batch_size:
                current_state, current_id, depth = queue.popleft()
                if current_id == end_id:
                    continue
                if max_steps is not None and depth >= max_steps:
                    continue
                batch.append((current_state, current_id, depth))

            if not batch:
                if not queue:
                    break
                continue

            futures: List[Tuple[concurrent.futures.Future, FullState, int, int]] = []
            for current_state, current_id, depth in batch:
                fut = executor.submit(
                    expand_state_pure,
                    current_state,
                    modplant_interfaces,
                    modplant_ops,
                    reaction_rules_df,
                )
                futures.append((fut, current_state, current_id, depth))

            for fut, current_state, current_id, depth in futures:
                try:
                    cands = fut.result()
                except Exception as e:
                    print(f"Error expanding state: {e}")
                    continue

                for from_state, to_state, op_str, cost, duration in cands:
                    if from_state not in visited:
                        new_from_id = len(state_list)
                        visited[from_state] = new_from_id
                        state_list.append(from_state)
                        queue.append((from_state, new_from_id, depth + 1))
                    from_id = visited[from_state]

                    if to_state not in visited:
                        new_to_id = len(state_list)
                        visited[to_state] = new_to_id
                        state_list.append(to_state)
                        queue.append((to_state, new_to_id, depth + 1))
                    to_id = visited[to_state]

                    add_transition(from_id, from_state, op_str, to_id, to_state, cost, duration)

    return transition_list, state_list

current_threads = 0
current_batch_size = 0

import os
cpu_cores = os.cpu_count() or 1

dynamic_threads = max(1, cpu_cores)
if dynamic_threads > 10:
    dynamic_threads -= 2 # Reserve 2 cores for OS if many cores are available

if sys.platform == "win32":
    print("Detected OS: Windows")
    current_threads = dynamic_threads
    current_batch_size = current_threads * 64 if dynamic_threads >= 14 else current_threads * 32
elif sys.platform == "darwin":
    print("Detected OS: macOS")
    current_threads = cpu_cores
    current_batch_size = current_threads * 8
else:
    current_threads = dynamic_threads
    current_batch_size = current_threads * 8

print(f"Using {current_threads} threads with batch size {current_batch_size}")

mix_rpms = set()
for _, row in reaction_rules_df.iterrows():
    if str(row.get('Reaction Type','')).strip() == 'Mix':
        param_str = str(row.get('Reaction Param','')).strip()
        if '/' in param_str:
            rpm_token = param_str.split('/',1)[0].strip().split()[0]
        else:
            rpm_token = param_str.split()[0] if param_str else None
        if rpm_token:
            mix_rpms.add(str(rpm_token))
available_rpms = {str(op[1]) for ops in modplant_ops.values() for op in ops if op[0] == 'Stirring'}
missing_mix_rpms = {rpm for rpm in mix_rpms if rpm not in available_rpms}
skip_bfs = bool(missing_mix_rpms)
if skip_bfs:
    print(f'Skipping BFS: missing Mix RPMs {sorted(missing_mix_rpms)} in all modplants')
    transition_list = []
    state_list = [start_state, end_state]
    state_index = {start_state: 0, end_state: 1}
    index_state = {0: start_state, 1: end_state}
    bfs_feasible = False
    operation_rows_fallback = [
        {'Step': 1, 'Operation': 'Unfeasible - missing RPM', 'Cost': 0.0, 'Duration (s)': 0.0},
        {'Step': 2, 'Operation': 'End', 'Cost': 0.0, 'Duration (s)': 0.0},
    ]
    _ = tools.display_dataframe_to_user(name='Operation Sequence Table', dataframe=pd.DataFrame(operation_rows_fallback))
else:
    transition_list, state_list = run_bfs(
        start_state,
        modplant_interfaces,
        max_steps=None,   # set to None for unlimited search depth if you like
        num_threads=current_threads,         # change this to control the number of worker threads
        batch_size=current_batch_size,       # None -> default = num_threads
    )

state_index = {state: idx for idx, state in enumerate(state_list)}
index_state = {idx: state for idx, state in enumerate(state_list)}

df = pd.DataFrame(
    [
        {
            "From_ID": fid,
            "From_State": str(fstate),
            "Operation": op,
            "Cost": cost,
            "Duration": duration,
            "To_ID": tid,
            "To_State": str(tstate),
        }
        for fid, fstate, op, tid, tstate, cost, duration in transition_list
    ]
)

_ = tools.display_dataframe_to_user(name="Transition List", dataframe=df)

end_idx = state_index.get(end_state, -1)
has_end_transition = any(
    op.startswith("End,") and tid == end_idx
    for _, _, op, tid, _, _, _ in transition_list
)

bfs_feasible = has_end_transition and len(transition_list) > 0
operation_rows_fallback = None

if not bfs_feasible:
    operation_rows_fallback = [
        {"Step": 1, "Operation": "Unfeasible", "Cost": 0.0, "Duration (s)": 0.0},
        {"Step": 2, "Operation": "End", "Cost": 0.0, "Duration (s)": 0.0},
    ]
    _ = tools.display_dataframe_to_user(name="Operation Sequence Table", dataframe=pd.DataFrame(operation_rows_fallback))
    print("BFS found no feasible solution; falling back to stub sequence.")


Detected OS: macOS
Using 8 threads with batch size 64


**Transition List**

,From_ID,From_State,Operation,Cost,Duration,To_ID,To_State
0,0,"(('HC10', ('content', ('A : 10.0 litre',)), ('...",Connect(HC10_Out1 -> HC40_In1) for A,2.0,3.0,2,"(('HC10', ('content', ('A : 10.0 litre',)), ('..."
1,2,"(('HC10', ('content', ('A : 10.0 litre',)), ('...",Variable Transfer setup (HC10->HC40) for A: x ...,0.0,0.0,3,"(('HC10', ('content', ('A: 10.0 + -1.0 x litre..."
2,0,"(('HC10', ('content', ('A : 10.0 litre',)), ('...",Connect(HC20_Out1 -> HC40_In1) for B,2.0,3.0,4,"(('HC10', ('content', ('A : 10.0 litre',)), ('..."
3,4,"(('HC10', ('content', ('A : 10.0 litre',)), ('...",Variable Transfer setup (HC20->HC40) for B: x ...,0.0,0.0,5,"(('HC10', ('content', ('A : 10.0 litre',)), ('..."
4,0,"(('HC10', ('content', ('A : 10.0 litre',)), ('...",Connect(HC30_Out1 -> HC40_In1) for C,2.0,3.0,6,"(('HC10', ('content', ('A : 10.0 litre',)), ('..."
...,...,...,...,...,...,...,...
272825,144653,"(('HC10', ('content', ('C: 7.0 litre',)), ('ou...","Separation: Open Valve of HC30_Out1 only, Drai...",3.0,0.0,144674,"(('HC10', ('content', ('C: 10.0 litre',)), ('o..."
272826,144655,"(('HC10', ('content', ('C: 7.0 litre',)), ('ou...","Separation: Open Valve of HC30_Out1 only, Drai...",3.0,0.0,144675,"(('HC10', ('content', ('C: 10.0 litre',)), ('o..."
272827,144657,"(('HC10', ('content', ('C: 7.0 litre',)), ('ou...","Separation: Open Valve of HC30_Out1 only, Drai...",3.0,0.0,144676,"(('HC10', ('content', ('C: 10.0 litre',)), ('o..."
272828,144666,"(('HC10', ('content', ('A: 9.00 litre',)), ('o...","Separation: Open Valve of HC30_Out1 only, Drai...",3.0,0.0,144677,"(('HC10', ('content', ('A: 9.00 litre',)), ('o..."


## OPT over the Reachable-State Graph

Here, **OPT** means the **optimizer / optimization-solving stage** of ModPlant-OPT.

Once BFS has produced the reachable-state graph, the notebook lifts it into a binary path-selection model in Pyomo. `trans_data[i] = (from_id, to_id, operation_label, cost, duration)` is the edge representation consumed by the model, and `x[i]` indicates whether the corresponding transition belongs to the selected execution path. This section is the notebook's explicit realization of the paper's **OPT** layer.


In [97]:
if not bfs_feasible:
    model = None
    trans_data = {}
    print("BFS infeasible: skipping model/optimization; proceeding directly to export.")
else:
    from pyomo.environ import ConcreteModel, RangeSet, Var, Binary
    model = ConcreteModel(name="ModPlant_Operation_Planner")
    model.S = RangeSet(0, len(state_list) - 1)
    model.T = RangeSet(0, len(transition_list) - 1)
    model.x = Var(model.T, domain=Binary)

    trans_data = {
        i: (fid, tid, op, cost, duration)
        for i, (fid, fstate, op, tid, tstate, cost, duration) in enumerate(transition_list)
    }


## Objective Function

The optimization layer maximizes the scalar objective implemented in `total_profit(m)`:

$$
\max \; P_0 - \sum_{t \in \mathcal{T}} x_t c_t + \lambda \sum_{t \in \mathcal{T}} x_t d_t
$$

Here `P_0 = order_profit_factor[0]` is the fixed revenue term, `c_t` is the transition cost, `d_t` is the transition duration, and `\lambda = order_profit_factor[1]` is the signed delay coefficient. In the default reference case, `\lambda < 0`, so longer plans reduce the objective value.


In [98]:
from pyomo.environ import Objective, maximize

if model is not None:
    def total_profit(m):
        total_cost = sum(model.x[t] * trans_data[t][3] for t in model.T)
        total_duration = sum(model.x[t] * trans_data[t][4] for t in model.T)
        fixed_profit = order_profit_factor[0]
        delay_penalty = total_duration * order_profit_factor[1]
        return fixed_profit - total_cost + delay_penalty

    model.obj = Objective(rule=total_profit, sense=maximize)
else:
    print("Objective skipped because model is None (BFS infeasible).")


## Flow Conservation and Boundary Conditions

The next constraint cell uses one `flow_rule` to encode all path-balance conditions. The start state acts as the unique source, the end state as the unique sink, and every intermediate state must satisfy inflow equals outflow.


### Single `flow_rule` Formulation

For each state $s \in \mathcal{S}$, the implemented constraint is written in the following piecewise form:
For each state $s \in \mathcal{S}$:

- If $s$ is the start state:
  $$
  \text{outflow}(s) - \text{inflow}(s) = \sum_{\substack{t \in \mathcal{T} \\ \text{src}(t) = s}} x_t - \sum_{\substack{t \in \mathcal{T} \\ \text{dst}(t) = s}} x_t = 1
  $$
- If $s$ is the goal state (i.e., contains only the goal product and is in the goal ModPlant):
  $$
  \text{inflow}(s) - \text{outflow}(s) = \sum_{\substack{t \in \mathcal{T} \\ \text{dst}(t) = s}} x_t - \sum_{\substack{t \in \mathcal{T} \\ \text{src}(t) = s}} x_t = 1
  $$
- Otherwise:
  $$
  \text{inflow}(s) - \text{outflow}(s) = \sum_{\substack{t \in \mathcal{T} \\ \text{dst}(t) = s}} x_t - \sum_{\substack{t \in \mathcal{T} \\ \text{src}(t) = s}} x_t = 0
  $$

This single conservation rule therefore subsumes both the boundary conditions and the continuity requirement of the selected path.


In [99]:
from collections import defaultdict

if model is not None:
    in_edges = defaultdict(list)
    out_edges = defaultdict(list)

    for t in model.T:
        from_state, to_state, *_ = trans_data[t]
        in_edges[to_state].append(t)
        out_edges[from_state].append(t)

    start_idx = state_index[start_state]
    end_idx = state_index[end_state]

    def flow_rule(m, s):
        inflow = sum(m.x[t] for t in in_edges[s])
        outflow = sum(m.x[t] for t in out_edges[s])
        if s == start_idx:
            return outflow - inflow == 1
        elif s == end_idx:
            return inflow - outflow == 1
        else:
            return inflow - outflow == 0

    model.flow = Constraint(model.S, rule=flow_rule)
else:
    print("Flow constraints skipped because model is None (BFS infeasible).")


## Solve and Display Results

The following cell adds a simple path-length cap, solves the Pyomo model with GLPK, and reports the optimal objective value when the reachable-state model is feasible.


In [100]:
def solve_and_display(model):
    model.max_steps = Constraint(expr=sum(model.x[t] for t in model.T) <= 1000)
    solver_name = "glpk"
    solver = SolverFactory(solver_name)

    if solver is None or not solver.available(exception_flag=False):
        raise RuntimeError(
            f"Selected solver '{solver_name}' was not detected. "
            "Please follow the installation instructions in README.md and restart your computer."
        )

    try:
        results = solver.solve(model, tee=True)
    except Exception as e:
        raise RuntimeError(
            f"Selected solver '{solver_name}' could not be started. "
            "Please follow the installation instructions in README.md and restart your computer. "
            f"Original error: {e}"
        ) from e

    if (
        results.solver.termination_condition == TerminationCondition.optimal
        and str(results.solver.status).lower() == 'ok'
    ):
        df_result = pd.DataFrame([{"Optimal Objective Value": value(model.obj)}])
        _ = tools.display_dataframe_to_user(name="Optimal Objective Value", dataframe=df_result)
    else:
        raise RuntimeError("Problem is infeasible or unsolvable.")

if model is not None:
    solve_and_display(model)
else:
    print("Skipping solver because BFS was infeasible; exporting corpus directly.")


GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /var/folders/xv/vvstx0bs44zbr_r9l19pk3gw0000gn/T/tmp1u5p876_.glpk.raw
 --wglp /var/folders/xv/vvstx0bs44zbr_r9l19pk3gw0000gn/T/tmp342gjzlo.glpk.glp
 --cpxlp /var/folders/xv/vvstx0bs44zbr_r9l19pk3gw0000gn/T/tmpqkm1jbxy.pyomo.lp
Reading problem data from '/var/folders/xv/vvstx0bs44zbr_r9l19pk3gw0000gn/T/tmpqkm1jbxy.pyomo.lp'...
/var/folders/xv/vvstx0bs44zbr_r9l19pk3gw0000gn/T/tmpqkm1jbxy.pyomo.lp:1663193: warning: lower bound of variable 'x3' redefined
/var/folders/xv/vvstx0bs44zbr_r9l19pk3gw0000gn/T/tmpqkm1jbxy.pyomo.lp:1663193: warning: upper bound of variable 'x3' redefined
144680 rows, 272831 columns, 818490 non-zeros
272830 integer variables, all of which are binary
1936023 lines were read
Writing problem data to '/var/folders/xv/vvstx0bs44zbr_r9l19pk3gw0000gn/T/tmp342gjzlo.glpk.glp'...
1373830 lines were written
GLPK Integer Optimizer 5.0
144680 rows, 272831 columns, 818490 non-zeros
272830 integer 

**Optimal Objective Value**

,Optimal Objective Value
0,123.0


## Operation Sequence Extraction

This post-processing step reconstructs the selected path from the solved binary edge variables and reshapes it into the paper-facing trace representation. The implementation also applies the public artifact's formatting heuristics, such as merging symbolic setup and resolve steps into dosing descriptions, grouping connect steps first, and splitting the terminating `End` marker into its own row.


In [101]:
from pyomo.environ import value
if not bfs_feasible:
    operation_rows = operation_rows_fallback
else:
    import pandas as pd
    import re

    merge_setup_resolve = True

    separate_connect_first = True

    merge_consecutive_dosing = True

    split_end_step = True

    def _reformat_all_volumes(op_str: str) -> str:
        def repl(m):
            try:
                val = float(m.group(1))
                return f": {val:.1f} litre"
            except:
                return m.group(0)

        return re.sub(r":\s*(\d+\.?\d*)\s*litre", repl, op_str)

    def _scale_dosing_volume(op_str: str, factor: int) -> str:
        m = re.search(r"\(([^()]*)\)\s*$", op_str)
        if not m: return op_str
        inner = m.group(1)
        parts = inner.split(",")
        new_parts = []
        for p in parts:
            p = p.strip()
            if ":" not in p:
                new_parts.append(p)
                continue
            name, rest = p.split(":", 1)
            tokens = rest.strip().split()
            if not tokens:
                new_parts.append(p)
                continue
            try:
                qty = float(tokens[0])
            except ValueError:
                new_parts.append(p)
                continue
            new_qty = qty * factor

            tokens[0] = f"{new_qty:.1f}"
            new_rest = " ".join(tokens)
            new_parts.append(f"{name.strip()}: {new_rest}")
        new_inner = ", ".join(new_parts)
        return op_str[:m.start()] + f"({new_inner})"

    result_path = []
    for t in model.T:
        if value(model.x[t]) > 0.5:
            result_path.append(trans_data[t])

    path_by_flow = []
    if start_state in state_index:
        curr_idx = state_index[start_state]
        while True:
            found = False
            for entry in result_path:
                fid, tid, op_str, cost, duration = entry
                if fid == curr_idx:
                    path_by_flow.append((op_str, cost, duration))
                    curr_idx = tid
                    found = True
                    break
            if not found:
                break

    if merge_setup_resolve:
        merged_transfer_path = []
        skip_next = False
        n = len(path_by_flow)

        for i in range(n):
            if skip_next:
                skip_next = False
                continue

            op_str, cost, dur = path_by_flow[i]

            if "Variable Transfer setup" in op_str and i + 1 < n:
                next_op, next_cost, next_dur = path_by_flow[i+1]
                if "Resolve(x=" in next_op:
                    mat_match = re.search(r"for\s+([A-Za-z0-9_]+):", op_str)
                    mat_name = mat_match.group(1) if mat_match else "Transfer"

                    m_setup = re.search(r"\(([^)]+)->([^)]+)\)", op_str)
                    src, dst = (m_setup.group(1), m_setup.group(2)) if m_setup else ("??", "??")

                    m_x = re.search(r"x=([\d\.]+)", next_op)
                    val = float(m_x.group(1)) if m_x else 0.0

                    final_desc = f"Dosing: Open Valve of {src}_Out1 only, Draining({src}), Filling({dst}), ({mat_name}: {val:.1f} litre)"

                    merged_transfer_path.append((final_desc, next_cost, next_dur))
                    skip_next = True
                    continue

            merged_transfer_path.append((op_str, cost, dur))

        path_by_flow = merged_transfer_path

    if separate_connect_first:
        connect_ops = [x for x in path_by_flow if x[0].strip().startswith("Connect")]
        other_ops   = [x for x in path_by_flow if not x[0].strip().startswith("Connect")]
        path_by_flow = connect_ops + other_ops

    if merge_consecutive_dosing:
        merged_path = []
        i = 0
        n = len(path_by_flow)

        pattern_dosing = re.compile(r"Draining\(([^)]+)\),\s*Filling\(([^)]+)\),.*\(([^:]+):\s*([\d\.]+)\s*litre\)")

        while i < n:
            op_str, cost, duration = path_by_flow[i]

            if not op_str.strip().startswith("Dosing:"):
                merged_path.append((op_str, cost, duration))
                i += 1
                continue

            match = pattern_dosing.search(op_str)
            if not match:
                merged_path.append((op_str, cost, duration))
                i += 1
                continue

            src, dst, mat, vol_str = match.groups()
            current_vol = float(vol_str)
            total_cost = cost
            total_duration = duration

            j = i + 1
            while j < n:
                next_op, next_cost, next_dur = path_by_flow[j]

                if not next_op.strip().startswith("Dosing:"):
                    break

                next_match = pattern_dosing.search(next_op)
                if not next_match:
                    break

                n_src, n_dst, n_mat, n_vol_str = next_match.groups()

                if (n_src == src) and (n_dst == dst) and (n_mat == mat):
                    current_vol += float(n_vol_str)
                    total_cost += next_cost
                    total_duration += next_dur
                    j += 1
                else:
                    break

            if current_vol > 1e-6:
                new_op_str = re.sub(
                    r":\s*[\d\.]+\s*litre",
                    f": {current_vol:.1f} litre",
                    op_str,
                    count=1
                )
                merged_path.append((new_op_str, total_cost, total_duration))

            i = j

        path_by_flow = merged_path

    if split_end_step:
        final_split_path = []
        for op_str, cost, dur in path_by_flow:
            if op_str.strip().startswith("End, "):
                cleaned_op = op_str.replace("End, ", "", 1).strip()
                final_split_path.append((cleaned_op, cost, dur))
                final_split_path.append(("End", 0.0, 0.0))
            else:
                final_split_path.append((op_str, cost, dur))
        path_by_flow = final_split_path

    final_formatted_path = []
    for op_str, cost, dur in path_by_flow:
        new_op_str = _reformat_all_volumes(op_str)
        final_formatted_path.append((new_op_str, cost, dur))
    path_by_flow = final_formatted_path

    operation_rows = []
    for idx, (op_str, cost, duration) in enumerate(path_by_flow):
        operation_rows.append({
            "Step": idx + 1,
            "Operation": op_str,
            "Cost": cost,
            "Duration (s)": duration,
        })

    df_result = pd.DataFrame(operation_rows)
    _ = tools.display_dataframe_to_user(name="Operation Sequence Table", dataframe=df_result)


**Operation Sequence Table**

,Step,Operation,Cost,Duration (s)
0,1,Connect(HC20_Out1 -> HC40_In1) for B,2.0,3.0
1,2,Connect(HC40_Out1 -> HC20_In1) for B,2.0,3.0
2,3,Connect(HC10_Out1 -> HC20_In2) for A,2.0,3.0
3,4,Connect(HC30_Out1 -> HC20_In3) for C,2.0,3.0
4,5,Connect(HC20_Out2 -> HC30_In1) for C,2.0,3.0
5,6,Connect(HC20_Out3 -> HC10_In1) for A,2.0,3.0
6,7,"Dosing: Open Valve of HC20_Out1 only, Draining...",30.0,100.0
7,8,"Dosing: Open Valve of HC10_Out1 only, Draining...",3.0,10.0
8,9,"Dosing: Open Valve of HC40_Out1 only, Draining...",6.0,20.0
9,10,"Dosing: Open Valve of HC30_Out1 only, Draining...",9.0,30.0


## Structured Artifact Export

The final cell writes the selected execution trace, plant context, and reaction-rule table to a JSONL corpus under `Result/Feasible` or `Result/Unfeasible`. This export is the machine-readable artifact retained for downstream analysis; the public notebook does not generate a state-transition graph visualization in its current form.


In [102]:
import json
import re
from pathlib import Path
import pandas as pd

if use_original_modplants:
    random_seed = 0
corpus_path = Path(f'Result/Feasible/run-{random_seed}-operation_sequence_corpus.jsonl') if bfs_feasible else Path(f'Result/Unfeasible/run-{random_seed}-operation_sequence_corpus_fallback.jsonl')
corpus_path.parent.mkdir(parents=True, exist_ok=True)

corpus_context = {
    'modplant_ops': modplant_ops,
    'modplant_interfaces': modplant_interfaces,
    'modplant_maximum_volume': modplant_maximum_volume,
    'modplant_resources': modplant_resources,
    'use_lazy_transfer_states': use_lazy_transfer_states,
    'min_transfer_volume': min_transfer_volume,
    'order': order,
    'reaction_rules': reaction_rules_df.to_dict(orient='records'),
}

_op_prefix_re = re.compile(r'^([A-Za-z]+)')
_qty_re = re.compile(r'([A-Za-z0-9_]+):\s*([0-9.]+)\s*litre')
_port_re = re.compile(r'Open Valve of ([A-Za-z0-9_]+)_Out\d+.*Filling\(([^)]+)\)')

def normalize_op_type(op_str: str) -> str:
    m = _op_prefix_re.match(op_str.strip())
    if m:
        return m.group(1)
    return op_str.strip().split()[0]

def parse_step(row):
    op_str = row['Operation']
    op_type = normalize_op_type(op_str)
    mats = [
        {'name': m, 'qty_l': float(q)}
        for m, q in _qty_re.findall(op_str)
    ]
    ports = {}
    pm = _port_re.search(op_str)
    if pm:
        ports = {'src': pm.group(1), 'dst': pm.group(2)}
    return {
        'role': 'step',
        'step_id': int(row['Step']),
        'op_type': op_type,
        'op_str': op_str,
        'materials': mats,
        'ports': ports,
        'cost': float(row['Cost']),
        'duration_s': float(row['Duration (s)']),
    }

trajectory_id = f"run-{random_seed}-{pd.Timestamp.now().strftime('%Y%m%d-%H%M%S')}"

records = []
records.append({'role': 'context', **corpus_context})
records.append({'role': 'metadata', 'trajectory_id': trajectory_id, 'bfs_feasible': bfs_feasible, 'use_lazy_transfer_states': use_lazy_transfer_states, 'min_transfer_volume': min_transfer_volume})

for row in operation_rows:
    records.append(parse_step(row))

with corpus_path.open('w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f"Saved structured result corpus to {corpus_path} with {len(records)} records")
#tools.display_dataframe_to_user(name='Result Corpus Preview', dataframe=pd.DataFrame(records))


Saved structured result corpus to Result/Feasible/run-0-operation_sequence_corpus.jsonl with 19 records
